# 🔧 Phase 4 Debug — Scan → Lock Transition
Self-contained notebook. Run cells 1→2→3→4→5→6→7→8 in order.
**Goal:** confirm port 5066 stays open during lock and monitor works.

## Cell 1 — Configuration

In [1]:
import sys, pathlib

RP_IP    = "192.168.0.99"
RP_KEY   = "Cav"
SSH_USER = "root"
SSH_PASS = "root"

# Scan params
CAV_DEC       = 32                            # change scan period
CAV_AMP       = 0.38                           # V — scan amplitude
CAV_OFFSET    = 0.4                           # V — scan offset
CAV_RANGE     = [[2.47, 2.53], [2.88, 2.97]] # ms — reference peak windows
CAV_LOCKPOINT = 2.93                          # ms — lockpoint
CAV_PID       = {"P": 0.0, "I": 2.0, "D": 0.0, "I_val": 0, "limit": [-0.99, 0.99]}

SHOW_TRIGGER  = True

_CAV_PERIOD_MS = 8e-9 * 16384 * CAV_DEC * 1e3

# Locate repo root
_here = pathlib.Path().resolve()
for _p in [_here] + list(_here.parents)[:4]:
    if (_p / "lockclient.py").exists():
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        print("Repo root:", _p)
        break

print(f"Board     : {RP_IP}")
print(f"Scan      : amp={CAV_AMP}V  offset={CAV_OFFSET}V  dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f}ms")
print(f"Lock      : range={CAV_RANGE}  lp={CAV_LOCKPOINT}ms")

Repo root: C:\Users\RikteemBhowmick\Projects 2025\RedPitaya Projects\RP-STCL
Board     : 192.168.0.99
Scan      : amp=0.38V  offset=0.4V  dec=32  period=4.194ms
Lock      : range=[[2.47, 2.53], [2.88, 2.97]]  lp=2.93ms


## Cell 2 — Connect (uploads RP_Lock.py, starts board)

In [2]:
import threading, time
from lockclient import LockClient, RP_client, Monitor

Monitor.show_trigger = SHOW_TRIGGER

# Clean up any stale session
try:
    if "Lock" in dir() and Lock is not None:
        try: Lock.close()
        except: pass
        # Reset stale lsock on all RPs
        for rp in Lock.RPs.values():
            rp.lsock = None
            rp.loop_running = False
    time.sleep(1)
except: pass

RPs = {"Cav": RP_client((RP_IP, 5000), {}, mode="scan_mon")}

print("Uploading and connecting...")
Lock = LockClient(RPs)

err = {}
def _connect():
    try: Lock.connect_all()
    except Exception as e: err["e"] = e

t = threading.Thread(target=_connect, daemon=True)
t.start(); t.join(timeout=45)
if t.is_alive():   raise TimeoutError("connect_all timed out")
if "e" in err:     raise RuntimeError(f"connect_all failed: {err['e']}")
print("Connected OK")

stcl_thread = threading.Thread(target=Lock.start, daemon=True)
stcl_thread.start()
time.sleep(2)

Lock.set_dec("Cav", CAV_DEC)
print(f"Event loop started  dec={CAV_DEC}  period={_CAV_PERIOD_MS:.3f}ms")

Uploading and connecting...
connecting...
Connected OK
Event loop started  dec=32  period=4.194ms


## Cell 3 — Push settings + start scan

In [3]:
import time

Lock.update_setting("Cav", "Master", "range",     CAV_RANGE)
Lock.update_setting("Cav", "Master", "lockpoint", CAV_LOCKPOINT)
Lock.update_setting("Cav", "Master", "enabled",   True)
Lock.update_setting("Cav", "Master", "PID",       CAV_PID)
print("Settings pushed")

Lock.start_scan("Cav", amplitude=CAV_AMP, offset=CAV_OFFSET)
print(f"Scan started — waiting 12s for board to open ports 5065 + 5066...")
time.sleep(12)
print("loop_running:", Lock.RPs["Cav"].loop_running)

check if lockpoint is still fine
Settings pushed
Scan started — waiting 12s for board to open ports 5065 + 5066...
connected to <socket.socket fd=2212, family=2, type=1, proto=0, laddr=('192.168.0.147', 52220), raddr=('192.168.0.99', 5065)>
loop_running: True


## Cell 4 — Verify port 5066 during SCAN

In [4]:
import socket, struct, json

def test_5066(label=""):
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(3)
        s.connect((RP_IP, 5066))
        s.sendall(bytes([0]))
        hdr = b""
        while len(hdr) < 4:
            hdr += s.recv(4 - len(hdr))
        length = struct.unpack(">I", hdr)[0]
        payload = b""
        while len(payload) < length:
            payload += s.recv(min(65536, length - len(payload)))
        s.close()
        dur, data = json.loads(payload.decode())
        print(f"✓ {label} port 5066 OK  dur={dur:.3f}ms  samples={len(data)}  range=[{min(data):.3f}, {max(data):.3f}]V")
        return True
    except Exception as e:
        print(f"✗ {label} port 5066 FAILED: {e}")
        return False

ok = test_5066("SCAN")
if not ok:
    print("\nBoard console might show the issue. Check Cell 8.")
    print("Common cause: RP_Lock.py on board still has old scan monitor server code.")

✓ SCAN port 5066 OK  dur=4.194ms  samples=16384  range=[-0.039, 0.073]V


## Cell 5 — Start monitor during scan (Qt window)

In [5]:
print("Starting monitor...")
Lock.start_monitor("Cav")
print("✓ Monitor started")
print("You should see cavity peaks sweeping. Adjust CAV_RANGE if peaks not in blue regions.")
print("When ready, run Cell 6 to transition to lock.")

Starting monitor...
Setting up monitor on main thread (Qt window)...
Settings added to plot
Monitor started (scan_mon mode — port 5066)
✓ Monitor started
You should see cavity peaks sweeping. Adjust CAV_RANGE if peaks not in blue regions.
When ready, run Cell 6 to transition to lock.


Settings added to plot


## Cell 6 — Scan → Lock transition (the critical step)

In [6]:
import time, socket, threading

print("="*50)
print("Step 1: Stopping scan loop...")
Lock.stop_loop("Cav")
time.sleep(1)
print(f"  loop_running: {Lock.RPs['Cav'].loop_running}")

# Check port 5066 closed
try:
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(1); s.connect((RP_IP, 5066)); s.close()
    print("  port 5066: still open (OK — _mon_stop may not have fired yet)")
except:
    print("  port 5066: closed ✓")

print("\nStep 2: Starting lock (in background thread)...")
_lock_done = threading.Event()
def _run_lock():
    Lock.start_lock("Cav")
    _lock_done.set()
threading.Thread(target=_run_lock, daemon=True).start()

print("\nStep 3: Polling port 5066 every second (up to 20s)...")
_port_ok = False
for i in range(20):
    time.sleep(1)
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(1); s.connect((RP_IP, 5066)); s.close()
        print(f"  t={i+1}s: port 5066 OPEN ✓")
        _port_ok = True
        break
    except Exception as e:
        print(f"  t={i+1}s: not yet ({type(e).__name__})")

print()
if _port_ok:
    print("✓ Port 5066 is open during lock — run Cell 7")
else:
    print("✗ Port 5066 never opened during lock")
    print("  _lock_monitor_server not starting on board")
    print("  → Check board console (Cell 8)")

Step 1: Stopping scan loop...
  loop_running: False
  port 5066: still open (OK — _mon_stop may not have fired yet)

Step 2: Starting lock (in background thread)...

Step 3: Polling port 5066 every second (up to 20s)...
  t=1s: port 5066 OPEN ✓

✓ Port 5066 is open during lock — run Cell 7


## Cell 7 — Test port 5066 during lock + restart monitor

In [7]:
import time

# Test data
ok = test_5066("LOCK")

if ok:
    print("\nStarting monitor during lock...")
    # Stop old monitor first (its update thread is dead from port closing)
    Lock.stop_monitor("Cav")
    time.sleep(0.5)
    Lock.start_monitor("Cav")
    print("\n✓ Monitor running during lock!")
    print("  LOCKED   = peaks stationary on amber lockpoint line")
    print("  UNLOCKED = peaks still drifting")
else:
    print("\nPort 5066 not working during lock.")
    print("Check Cell 8 for board errors.")

✓ LOCK port 5066 OK  dur=4.194ms  samples=16384  range=[-0.059, 0.000]V

Starting monitor during lock...
Setting up monitor on main thread (Qt window)...
connected to <socket.socket fd=2592, family=2, type=1, proto=0, laddr=('192.168.0.147', 52252), raddr=('192.168.0.99', 5065)>
Settings added to plot
Monitor started (scan_mon mode — port 5066)

✓ Monitor running during lock!
  LOCKED   = peaks stationary on amber lockpoint line
  UNLOCKED = peaks still drifting


## Cell 8 — Board diagnostics (run anytime)

In [8]:
import paramiko

ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(RP_IP, port=22, username=SSH_USER, password=SSH_PASS)

print("=== Open ports ===")
_, out, _ = ssh.exec_command("ss -tlnp | grep -E '5000|5065|5066'")
print(out.read().decode().strip() or "  (none)")

print("\n=== Key lines in RP_Lock.py on board ===")
_, out, _ = ssh.exec_command(
    "grep -n '_mon_stop\|_lock_mon_stop\|ch_byte\[0\]\|while not\|srv.bind\|5066' /root/RP_Lock.py")
print(out.read().decode().strip())

print("\n=== Running python processes ===")
_, out, _ = ssh.exec_command("ps aux | grep python | grep -v grep")
print(out.read().decode().strip())

ssh.close()

=== Open ports ===
LISTEN 0      128     192.168.0.99:5000      0.0.0.0:*    users:(("python3",pid=766,fd=7))                       
LISTEN 1      128     192.168.0.99:5065      0.0.0.0:*    users:(("python3",pid=766,fd=12))                      
LISTEN 0      5            0.0.0.0:5066      0.0.0.0:*    users:(("python3",pid=766,fd=10))

=== Key lines in RP_Lock.py on board ===
395:            # Start port 5066 monitor server so the PC-side Monitor can keep
398:            _lock_mon_stop = _thr.Event()
403:                srv.bind(("", 5066))   # all interfaces
406:                print("Lock monitor server listening on port 5066")
407:                while not _lock_mon_stop.is_set():
415:                        ch = ch_byte[0] if ch_byte else 0
433:            _lock_mon_stop.clear()
435:            print("Lock monitor server started on port 5066")
438:            _lock_mon_stop.set()  # stop monitor server
506:            Dedicated monitor server on port 5066.
514:            srv.bin

## Cell 9 — Stop everything

In [9]:
import time
try: Lock.stop_monitor("Cav")
except: pass
time.sleep(0.5)
try: Lock.stop_loop("Cav")
except: pass
time.sleep(0.5)
try: Lock.close()
except: pass
print("All stopped.")

All stopped.
